## Прогноз виручки (three-phase linear)

Щомісячний прогноз виручки по категоріях із повторним використанням пайплайну.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from three_phase_linear import ForecastConfig, run_three_phase_forecast

DATA_PATH = Path('forecast_of_market_dataset.csv')
OUTPUT_PATH = Path('money_three_phase_forecast_accuracy_calculation.csv')
GROUP_COLS = ['category_id']
TARGET_COLUMN = 'revenue'
REGRESSORS = ['is_sale_prohibition', 'cos_month', 'sin_month', 'cos_quarter', 'sin_quarter', 'unique_brand_count']


In [3]:
# Try to read with the semicolon first, but fall back to comma if the file was parsed as a single column
df = pd.read_csv(DATA_PATH, sep=';', engine='python')
if len(df.columns) == 1:
    df = pd.read_csv(DATA_PATH, sep=',', engine='python')

# Normalize common column names in the source data to the notebook's expected names
if 'month' in df.columns and 'date' not in df.columns:
    df = df.rename(columns={'month': 'date'})
if 'product_group_id' in df.columns and 'category_id' not in df.columns:
    df = df.rename(columns={'product_group_id': 'category_id'})
if 'product_group_name' in df.columns and 'category_title' not in df.columns:
    df = df.rename(columns={'product_group_name': 'category_title'})

# If the expected TARGET_COLUMN does not exist, try common alternatives
if TARGET_COLUMN not in df.columns:
    if 'market_revenue' in df.columns:
        df[TARGET_COLUMN] = pd.to_numeric(df['market_revenue'], errors='coerce')
    elif 'revenue_amazon' in df.columns:
        df[TARGET_COLUMN] = pd.to_numeric(df['revenue_amazon'], errors='coerce')
    else:
        df[TARGET_COLUMN] = np.nan

# Ensure the date column exists and parse it
if 'date' in df.columns:
    df['date'] = pd.to_datetime(df['date'], dayfirst=True, errors='coerce')
else:
    # create an empty date column to avoid KeyError downstream
    df['date'] = pd.NaT

# Ensure grouping columns exist to avoid KeyError in sort/groupby
for c in GROUP_COLS + ['category_title']:
    if c not in df.columns:
        df[c] = np.nan

df = df.sort_values(GROUP_COLS + ['date']).reset_index(drop=True)

# Only coerce regressors that exist; create missing regressors with NaN so later code can rely on their presence
for col in REGRESSORS:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    else:
        df[col] = np.nan

agg_df = df.groupby(['date', 'category_id', 'category_title'], as_index=False).agg(
    revenue_sum=(TARGET_COLUMN, 'sum'),
    revenue_count=(TARGET_COLUMN, 'count'),
    is_sale_prohibition=('is_sale_prohibition', 'max'),
    cos_month=('cos_month', 'mean'),
    sin_month=('sin_month', 'mean'),
    cos_quarter=('cos_quarter', 'mean'),
    sin_quarter=('sin_quarter', 'mean'),
    unique_brand_count=('unique_brand_count', 'mean'),
)
agg_df.loc[agg_df['revenue_count'] == 0, 'revenue_sum'] = np.nan
df = agg_df.rename(columns={'revenue_sum': 'revenue'}).drop(columns=['revenue_count'])
df = df.sort_values(GROUP_COLS + ['date']).reset_index(drop=True)

future_mask = df[TARGET_COLUMN].isna()
future_counts = df[future_mask].groupby(GROUP_COLS).size()
forecast_horizon = int(future_counts.max()) if not future_counts.empty else 12
if forecast_horizon <= 0:
    forecast_horizon = 12

history_df = df[~future_mask].copy()
history_df = history_df.sort_values(GROUP_COLS + ['date']).reset_index(drop=True)
history_df['is_evaluation_period'] = False

for _, group in history_df.groupby(GROUP_COLS):
    eval_count = min(len(group), forecast_horizon)
    if eval_count == 0:
        continue
    eval_indices = group.tail(eval_count).index
    history_df.loc[eval_indices, 'is_evaluation_period'] = True

history_df['revenue_actual'] = history_df[TARGET_COLUMN]
history_df.loc[history_df['is_evaluation_period'], TARGET_COLUMN] = np.nan

df = history_df

print(f'Forecast horizon for accuracy: {forecast_horizon} periods')


Forecast horizon for accuracy: 12 periods


In [4]:
input_cols = ['date', *GROUP_COLS, TARGET_COLUMN, *REGRESSORS]
input_cols = list(dict.fromkeys(input_cols))
config = ForecastConfig(
    time_col='date',
    target_col=TARGET_COLUMN,
    group_cols=GROUP_COLS,
    freq='MS',
    forecast_horizon=forecast_horizon,
    seasonal_periods=12,
    min_history=24,
    lags=(1, 2, 3, 6, 12, 18, 24),
    rolling_windows=(3, 6, 12, 24),
    additional_regressors=REGRESSORS,
    random_search_iterations=10,
    n_splits=4,
    random_state=46,
)

preds, summaries = run_three_phase_forecast(df[input_cols].copy(), config)
preds = preds.rename(columns={
    'prediction': 'revenue_forecast',
    f'{TARGET_COLUMN}_holtwinters': 'revenue_baseline',
})
summary_report = pd.DataFrame({
    'group_key': [s.group_key[0] for s in summaries],
    'train_rows': [s.train_rows for s in summaries],
    'cv_mae': [s.best_score for s in summaries],
    'skipped_reason': [s.skipped_reason for s in summaries],
})
summary_report.head()

c:\Users\User\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
c:\Users\User\anaconda3\Lib\site-packages\statsmodels\tsa\holtwinters\model.py:918: ConvergenceWarning: Optimization failed to converge. Check mle_retvals.
  warnings.warn(
c:\Users\User\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
c:\Users\User\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(
c:\Users\User\anaconda3\Lib\site-packages\statsmodels\tsa\base\tsa_model

,group_key,train_rows,cv_mae,skipped_reason
0,1,60,1.639485e+09,None
1,2,60,1.031719e+09,None
2,3,60,2.233790e+08,None
3,4,60,1.030755e+09,None
4,5,60,2.271931e+08,None


In [6]:
merge_cols = [*GROUP_COLS, 'date']
result_df = df.copy()
result_df['is_evaluation_period'] = result_df[TARGET_COLUMN].isna()
result_df = result_df.merge(preds[merge_cols + ['revenue_forecast']], on=merge_cols, how='left')
result_df['revenue'] = result_df['revenue'].astype(float)
result_df['revenue'] = result_df['revenue'].fillna(result_df['revenue_forecast'])

output_columns = ['date', 'category_id', 'category_title', 'revenue']
final_output = result_df.loc[result_df['is_evaluation_period'], output_columns]
final_output = final_output.groupby(['date', 'category_id', 'category_title'], as_index=False)['revenue'].sum()
final_output = final_output.sort_values(['category_id', 'date']).reset_index(drop=True)
final_output.to_csv(OUTPUT_PATH, index=False)

final_output.tail()


,date,category_id,category_title,revenue
343,2024-01-08,43,Communication Gadgets,1.662740e+09
344,2024-01-09,43,Communication Gadgets,1.494500e+09
345,2024-01-10,43,Communication Gadgets,1.561306e+09
346,2024-01-11,43,Communication Gadgets,1.420449e+09
347,2024-01-12,43,Communication Gadgets,1.503472e+09


MAE - Середня абсолютна помилка 

In [7]:
evaluation_df = result_df[result_df['is_evaluation_period']].copy()
actual = evaluation_df['revenue_actual'].astype(float)
forecast = evaluation_df['revenue_forecast'].astype(float)
mask = actual.notna() & forecast.notna()
if mask.any():
    mae = np.abs(actual[mask] - forecast[mask]).mean()
    print(f'MAE: {mae:.4f}')
else:
    print('MAE: not enough data to calculate')


MAE: 640011849.8563


MAPE-Середня абсолютна відсоткова помилка 

In [8]:
evaluation_df = result_df[result_df['is_evaluation_period']].copy()
actual = evaluation_df['revenue_actual'].astype(float)
forecast = evaluation_df['revenue_forecast'].astype(float)
mask = actual.notna() & forecast.notna() & (actual.replace(0, np.nan).notna())
if mask.any():
    mape = (np.abs((actual[mask] - forecast[mask]) / actual[mask]) * 100).mean()
    print(f'MAPE: {mape:.4f}%')
else:
    print('MAPE: not enough data to calculate')


MAPE: 15.5025%


WMAPE-Взважена середня абсолютна помилка 

In [9]:
evaluation_df = result_df[result_df['is_evaluation_period']].copy()
actual = evaluation_df['revenue_actual'].astype(float)
forecast = evaluation_df['revenue_forecast'].astype(float)
mask = actual.notna() & forecast.notna()
denominator = np.abs(actual[mask]).sum()
if mask.any() and denominator > 0:
    wmape = np.abs(actual[mask] - forecast[mask]).sum() / denominator * 100
    print(f'WMAPE: {wmape:.4f}%')
else:
    print('WMAPE: not enough data to calculate')


WMAPE: 15.9446%


In [ ]:
summary_report